<a href="https://colab.research.google.com/github/MostaryKhatun/AnthraxPaper/blob/main/Bootstrap_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!apt-get install -y r-base r-base-dev libcurl4-openssl-dev libssl-dev libxml2-dev > /dev/null 2>&1
!pip install rpy2 --quiet

%load_ext rpy2.ipython

The rpy2.ipython extension is already loaded. To reload it, use:
  %reload_ext rpy2.ipython


In [ ]:
%%R
if (!requireNamespace("BiocManager", quietly = TRUE)) install.packages("BiocManager")

req_bioc <- c("WGCNA", "impute", "preprocessCore", "GO.db", "sva")
for (p in req_bioc) {
  if (!requireNamespace(p, quietly = TRUE)) BiocManager::install(p, update = FALSE, ask = FALSE)
}

req_cran <- c("data.table", "ggplot2", "pheatmap", "RColorBrewer", "VennDiagram",
              "gridExtra", "png", "dendextend", "pvclust")
for (p in req_cran) {
  if (!requireNamespace(p, quietly = TRUE)) install.packages(p, repos = "https://cloud.r-project.org")
}

suppressPackageStartupMessages({
  library(WGCNA); library(data.table); library(ggplot2)
  library(pheatmap); library(RColorBrewer); library(VennDiagram)
  library(grid); library(gridExtra); library(png); library(dendextend)
  library(pvclust)
})

cat("All packages loaded successfully.\n")

All packages loaded successfully.


In [ ]:
%%R
if (!requireNamespace("data.table", quietly = TRUE)) install.packages("data.table", repos = "https://cloud.r-project.org")
if (!requireNamespace("pvclust", quietly = TRUE)) install.packages("pvclust", repos = "https://cloud.r-project.org")
if (!requireNamespace("ggplot2", quietly = TRUE)) install.packages("ggplot2", repos = "https://cloud.r-project.org")

suppressPackageStartupMessages({
  library(data.table); library(pvclust); library(ggplot2)
})

cat("Packages ready for bootstrap analysis.\n")

Packages ready for bootstrap analysis.


In [ ]:
%load_ext rpy2.ipython

The rpy2.ipython extension is already loaded. To reload it, use:
  %reload_ext rpy2.ipython


In [ ]:
%%R
############################################################
## Individual Hub Gene Bootstrap Analysis (CORRECTED FOR WIDE FORMAT)
## Input layout: rows = samples, columns = genes, LAST column = target (0/1)
## 5-6 hub genes x 2 figures each + 1 summary table
## Output: /content/drive/MyDrive/Anthraxpaper/bootstrap analysis
############################################################

suppressPackageStartupMessages({
  library(data.table); library(ggplot2)
})
options(stringsAsFactors = FALSE)

OUTDIR <- "/content/drive/MyDrive/Anthraxpaper/bootstrap analysis"
if (!dir.exists(OUTDIR)) dir.create(OUTDIR, recursive = TRUE)

TABLEDIR <- file.path(OUTDIR, "Tables")
if (!dir.exists(TABLEDIR)) dir.create(TABLEDIR, recursive = TRUE)

TRAIN_EXPR <- "/content/drive/MyDrive/Bootstrap anlysis/datasets/GSE33630_matrix_Transpose_labeled.csv"
TEST_EXPR  <- "/content/drive/MyDrive/Bootstrap anlysis/datasets/GSE76894_matrix_Transpose_labeled.csv"

hub_genes_top5 <- c("MAP2K7","PPARA","RBCK1","NFATC2","PLXNA4","STAMBP")

############################################################
## Load WIDE-format expression data
## rows = samples, columns = genes, LAST column = target (0/1)
## optionally the FIRST column may be a non-numeric sample ID
############################################################

read_expr_wide <- function(path){
  dt <- fread(path)

  # last column = target label
  target_col_idx <- ncol(dt)
  target_col_name <- names(dt)[target_col_idx]
  target_raw <- dt[[target_col_idx]]
  dt[[target_col_idx]] <- NULL

  df <- as.data.frame(dt)

  # detect whether the first column is a non-numeric sample ID rather than a gene
  first_col <- df[[1]]
  first_is_id <- is.character(first_col) ||
    all(is.na(suppressWarnings(as.numeric(as.character(first_col)))))

  if (first_is_id) {
    samp_ids <- make.unique(as.character(first_col))
    df[[1]] <- NULL
  } else {
    samp_ids <- paste0("Sample_", seq_len(nrow(df)))
  }

  # coerce every remaining column to numeric (defensive against stray text/NA)
  df[] <- lapply(df, function(col) suppressWarnings(as.numeric(as.character(col))))
  rownames(df) <- samp_ids

  target <- suppressWarnings(as.numeric(as.character(target_raw)))

  list(expr = df, target = target, target_col_name = target_col_name, sample_ids = samp_ids)
}

dataA <- read_expr_wide(TRAIN_EXPR)   # GSE33630
dataB <- read_expr_wide(TEST_EXPR)    # GSE76894

expr_A <- dataA$expr
expr_B <- dataB$expr

cat("GSE33630:", nrow(expr_A), "samples,", ncol(expr_A), "genes",
    "(target column:", dataA$target_col_name, ")\n")
cat("GSE76894:", nrow(expr_B), "samples,", ncol(expr_B), "genes",
    "(target column:", dataB$target_col_name, ")\n")

cat("\nGSE33630 target distribution:\n"); print(table(dataA$target, useNA = "ifany"))
cat("GSE76894 target distribution:\n"); print(table(dataB$target, useNA = "ifany"))

present_hub <- intersect(hub_genes_top5, intersect(colnames(expr_A), colnames(expr_B)))
cat("\nHub genes available in both datasets:", length(present_hub), "of", length(hub_genes_top5), "—",
    paste(present_hub, collapse=", "), "\n")
missing_hub <- setdiff(hub_genes_top5, present_hub)
if (length(missing_hub) > 0) cat("Missing:", paste(missing_hub, collapse=", "), "\n")

if (length(present_hub) == 0) {
  stop("No hub genes matched any column names. Check for symbol-case mismatches, ",
       "prefixes (e.g. 'X' prepended by R to numeric-looking names), or whitespace ",
       "in the CSV header — print colnames(expr_A)[1:20] to inspect.")
}

############################################################
## Extract hub gene columns only (still samples x genes)
############################################################

expr_A_hub <- expr_A[, present_hub, drop = FALSE]
expr_B_hub <- expr_B[, present_hub, drop = FALSE]

combinedCheck <- as.matrix(rbind(expr_A_hub, expr_B_hub))

cat("\nValue range check — min:", round(min(combinedCheck, na.rm = TRUE), 3),
    " max:", round(max(combinedCheck, na.rm = TRUE), 3), "\n")

############################################################
## Only apply log2 transform if data is:
## (a) all non-negative, AND (b) looks like raw intensity (max > 50)
## Otherwise assume it's already log-scale / normalized and skip transform
############################################################

hasNegative <- any(combinedCheck < 0, na.rm = TRUE)
looksRaw <- max(combinedCheck, na.rm = TRUE) > 50

if (hasNegative) {
  cat("Data contains negative values — already normalized/log-scale. Skipping log2 transform.\n")
} else if (looksRaw) {
  expr_A_hub[] <- log2(expr_A_hub + 1)
  expr_B_hub[] <- log2(expr_B_hub + 1)
  cat("Applied log2(x+1) transform (raw non-negative intensities detected)\n")
} else {
  cat("Data already appears log-scale and non-negative — no transform applied\n")
}

############################################################
## Combine into long-format data frame
## Status = target label (0 = Normal, 1 = Affected)
############################################################

longData <- do.call(rbind, lapply(present_hub, function(g){
  data.frame(
    Gene = g,
    Sample = c(rownames(expr_A_hub), rownames(expr_B_hub)),
    Dataset = c(rep("GSE33630", nrow(expr_A_hub)), rep("GSE76894", nrow(expr_B_hub))),
    Target = c(dataA$target, dataB$target),
    Expression = c(expr_A_hub[[g]], expr_B_hub[[g]])
  )
}))

longData$Status <- factor(longData$Target, levels = c(0, 1), labels = c("Normal", "Affected"))

## Remove any remaining NA/NaN/Inf rows or unmapped target labels, defensively
nBefore <- nrow(longData)
longData <- longData[is.finite(longData$Expression) & !is.na(longData$Status), ]
nAfter <- nrow(longData)
if (nBefore != nAfter) {
  cat("\nRemoved", nBefore - nAfter, "rows with non-finite expression or unmapped target values.\n")
}
if (nAfter == 0) stop("No valid rows remain after cleaning — check target coding (expects 0/1) and expression values.")

############################################################
## Bootstrap function — robust to small N and near-constant data
############################################################

bootstrap_mean_ci <- function(x, nboot = 2000, seed = 1){
  x <- x[is.finite(x)]
  n <- length(x)

  if (n < 2) {
    return(data.frame(Mean = ifelse(n == 1, x, NA), CI_lower = NA, CI_upper = NA, N = n))
  }

  set.seed(seed)
  boot_means <- replicate(nboot, mean(sample(x, n, replace = TRUE)))
  boot_means <- boot_means[is.finite(boot_means)]

  if (length(boot_means) == 0) {
    return(data.frame(Mean = mean(x), CI_lower = NA, CI_upper = NA, N = n))
  }

  data.frame(
    Mean = mean(x),
    CI_lower = quantile(boot_means, 0.025, na.rm = TRUE),
    CI_upper = quantile(boot_means, 0.975, na.rm = TRUE),
    N = n
  )
}

############################################################
## Loop over each hub gene: generate 2 figures + collect stats
## Figures compare Affected vs Normal, faceted by Dataset
############################################################

summaryList <- list()

for (g in present_hub) {

  geneData <- subset(longData, Gene == g)

  if (nrow(geneData) == 0) {
    cat("\nSkipping", g, "— no valid data after cleaning.\n")
    next
  }

  ##########################################################
  ## FIGURE 1 — Expression distribution: Affected vs Normal, by dataset
  ##########################################################

  p1 <- ggplot(geneData, aes(x = Status, y = Expression, fill = Status)) +
    geom_boxplot(alpha = 0.6, outlier.shape = NA, width = 0.5) +
    geom_jitter(width = 0.12, size = 2, alpha = 0.7, shape = 21, colour = "black") +
    facet_wrap(~ Dataset) +
    scale_fill_manual(values = c("Normal" = "dodgerblue3", "Affected" = "firebrick3")) +
    theme_classic(base_size = 14) +
    theme(legend.position = "none",
          plot.title = element_text(face = "bold", hjust = 0.5, size = 15)) +
    labs(title = paste0(g, " — Expression: Affected vs Normal"),
         x = "", y = "Expression")

  f1_path <- file.path(OUTDIR, paste0(g, "_Fig1_ExpressionDistribution.png"))
  ggsave(f1_path, plot = p1, width = 7, height = 5, dpi = 300, bg = "white")

  ##########################################################
  ## FIGURE 2 — Bootstrap 95% CI of mean expression, per Dataset x Status
  ##########################################################

  ciRows <- do.call(rbind, lapply(unique(geneData$Dataset), function(ds){
    do.call(rbind, lapply(c("Normal", "Affected"), function(st){
      x <- geneData$Expression[geneData$Dataset == ds & geneData$Status == st]
      ci <- bootstrap_mean_ci(x, seed = which(unique(geneData$Dataset) == ds) * 10 +
                                 ifelse(st == "Normal", 1, 2))
      data.frame(Dataset = ds, Status = st, ci)
    }))
  }))

  p2 <- ggplot(ciRows, aes(x = Status, y = Mean, fill = Status)) +
    geom_col(width = 0.5, colour = "black", alpha = 0.75) +
    geom_errorbar(aes(ymin = CI_lower, ymax = CI_upper), width = 0.15, linewidth = 0.8, na.rm = TRUE) +
    facet_wrap(~ Dataset) +
    scale_fill_manual(values = c("Normal" = "dodgerblue3", "Affected" = "firebrick3")) +
    theme_classic(base_size = 14) +
    theme(legend.position = "none",
          plot.title = element_text(face = "bold", hjust = 0.5, size = 15)) +
    labs(title = paste0(g, " — Bootstrap 95% CI of Mean Expression"),
         x = "", y = "Mean Expression \u00b1 95% CI")

  f2_path <- file.path(OUTDIR, paste0(g, "_Fig2_BootstrapCI.png"))
  ggsave(f2_path, plot = p2, width = 7, height = 5, dpi = 300, bg = "white")

  cat("Saved figures for", g, ":\n  -", f1_path, "\n  -", f2_path, "\n")

  summaryList[[g]] <- data.frame(
    Gene = g,
    Dataset = ciRows$Dataset,
    Status = ciRows$Status,
    N = ciRows$N,
    Mean_Expression = round(ciRows$Mean, 4),
    Bootstrap_CI_Lower = round(ciRows$CI_lower, 4),
    Bootstrap_CI_Upper = round(ciRows$CI_upper, 4)
  )
}

############################################################
## Combine all gene summaries into ONE master table
############################################################

masterSummary <- do.call(rbind, summaryList)
rownames(masterSummary) <- NULL

write.csv(masterSummary, file.path(TABLEDIR, "Table_HubGenes_Bootstrap_Summary.csv"), row.names = FALSE)

cat("\n============================================================\n")
cat("Master summary table:\n")
print(masterSummary)
cat("============================================================\n")

cat("\nALL OUTPUTS SAVED TO:", OUTDIR, "\n")
cat("Figures generated: 2 per gene x", length(present_hub), "genes =", 2*length(present_hub), "total\n")
cat("Table saved: Tables/Table_HubGenes_Bootstrap_Summary.csv\n")

GSE33630: 105 samples, 24503 genes (target column: target )
GSE76894: 103 samples, 8637 genes (target column: target )

GSE33630 target distribution:

  0   1 
  1 104 
GSE76894 target distribution:

  0   1 
  1 102 

Hub genes available in both datasets: 6 of 6 — MAP2K7, PPARA, RBCK1, NFATC2, PLXNA4, STAMBP 

Value range check — min: 3.671  max: 11.266 
Data already appears log-scale and non-negative — no transform applied
Saved figures for MAP2K7 :
  - /content/drive/MyDrive/Anthraxpaper/bootstrap analysis/MAP2K7_Fig1_ExpressionDistribution.png 
  - /content/drive/MyDrive/Anthraxpaper/bootstrap analysis/MAP2K7_Fig2_BootstrapCI.png 
Saved figures for PPARA :
  - /content/drive/MyDrive/Anthraxpaper/bootstrap analysis/PPARA_Fig1_ExpressionDistribution.png 
  - /content/drive/MyDrive/Anthraxpaper/bootstrap analysis/PPARA_Fig2_BootstrapCI.png 
Saved figures for RBCK1 :
  - /content/drive/MyDrive/Anthraxpaper/bootstrap analysis/RBCK1_Fig1_ExpressionDistribution.png 
  - /content/drive/MyD

In [15]:
%%R
############################################################
## Individual Hub Gene Bootstrap Analysis — PUBLICATION READY
## CORRECTED FOR WIDE-FORMAT INPUT:
##   rows = samples, columns = genes, LAST column = target (0/1)
## Each gene: TWO SEPARATE figures (A = distribution, B = bootstrap CI),
## grouped by Affected vs Normal, faceted by dataset.
## 6 hub genes -> 12 separated figures total.
## Plus one master overview figure with all hub genes.
## Output: /content/drive/MyDrive/Bootstrap anlysis
############################################################

if (!requireNamespace("patchwork", quietly = TRUE)) install.packages("patchwork", repos = "https://cloud.r-project.org")

suppressPackageStartupMessages({
  library(data.table); library(ggplot2); library(patchwork)
})
options(stringsAsFactors = FALSE)

OUTDIR <- "/content/drive/MyDrive/Bootstrap anlysis"
if (!dir.exists(OUTDIR)) dir.create(OUTDIR, recursive = TRUE)

TABLEDIR <- file.path(OUTDIR, "Tables")
if (!dir.exists(TABLEDIR)) dir.create(TABLEDIR, recursive = TRUE)

TRAIN_EXPR <- "/content/drive/MyDrive/Bootstrap anlysis/datasets/GSE33630_matrix_Transpose_labeled.csv"
TEST_EXPR  <- "/content/drive/MyDrive/Bootstrap anlysis/datasets/GSE76894_matrix_Transpose_labeled.csv"

## ---- UPDATED: 6 hub genes ----
hub_genes_top5 <- c("MAP2K7","PPARA","RBCK1","NFATC2","PLXNA4","STAMBP")

############################################################
## Publication theme — clean, journal-style
############################################################

theme_pub <- function(base_size = 13){
  theme_bw(base_size = base_size) +
    theme(
      panel.grid.minor = element_blank(),
      panel.grid.major.x = element_blank(),
      plot.title = element_text(face = "bold", size = base_size + 1, hjust = 0.5),
      axis.title = element_text(face = "bold", size = base_size),
      axis.text = element_text(size = base_size - 1, colour = "black"),
      legend.position = "none",
      plot.margin = margin(10, 14, 10, 10)
    )
}

############################################################
## Load WIDE-format expression data
## rows = samples, columns = genes, LAST column = target (0/1)
## optionally the FIRST column may be a non-numeric sample ID
############################################################

read_expr_wide <- function(path){
  dt <- fread(path)

  target_col_idx <- ncol(dt)
  target_col_name <- names(dt)[target_col_idx]
  target_raw <- dt[[target_col_idx]]
  dt[[target_col_idx]] <- NULL

  df <- as.data.frame(dt)

  first_col <- df[[1]]
  first_is_id <- is.character(first_col) ||
    all(is.na(suppressWarnings(as.numeric(as.character(first_col)))))

  if (first_is_id) {
    samp_ids <- make.unique(as.character(first_col))
    df[[1]] <- NULL
  } else {
    samp_ids <- paste0("Sample_", seq_len(nrow(df)))
  }

  df[] <- lapply(df, function(col) suppressWarnings(as.numeric(as.character(col))))
  rownames(df) <- samp_ids

  target <- suppressWarnings(as.numeric(as.character(target_raw)))

  list(expr = df, target = target, target_col_name = target_col_name, sample_ids = samp_ids)
}

dataA <- read_expr_wide(TRAIN_EXPR)   # GSE33630
dataB <- read_expr_wide(TEST_EXPR)    # GSE76894

expr_A <- dataA$expr
expr_B <- dataB$expr

cat("GSE33630:", nrow(expr_A), "samples,", ncol(expr_A), "genes",
    "(target column:", dataA$target_col_name, ")\n")
cat("GSE76894:", nrow(expr_B), "samples,", ncol(expr_B), "genes",
    "(target column:", dataB$target_col_name, ")\n")

cat("\nGSE33630 target distribution:\n"); print(table(dataA$target, useNA = "ifany"))
cat("GSE76894 target distribution:\n"); print(table(dataB$target, useNA = "ifany"))

present_hub <- intersect(hub_genes_top5, intersect(colnames(expr_A), colnames(expr_B)))
cat("\nHub genes available in both datasets:", length(present_hub), "of", length(hub_genes_top5), "—",
    paste(present_hub, collapse=", "), "\n")
missing_hub <- setdiff(hub_genes_top5, present_hub)
if (length(missing_hub) > 0) cat("Missing:", paste(missing_hub, collapse=", "), "\n")

if (length(present_hub) == 0) {
  stop("No hub genes matched any column names. Check for symbol-case mismatches, ",
       "prefixes (e.g. 'X' prepended by R to numeric-looking names), or whitespace ",
       "in the CSV header — print colnames(expr_A)[1:20] to inspect.")
}

expr_A_hub <- expr_A[, present_hub, drop = FALSE]
expr_B_hub <- expr_B[, present_hub, drop = FALSE]
combinedCheck <- as.matrix(rbind(expr_A_hub, expr_B_hub))

cat("\nValue range — min:", round(min(combinedCheck, na.rm = TRUE), 3),
    " max:", round(max(combinedCheck, na.rm = TRUE), 3), "\n")

hasNegative <- any(combinedCheck < 0, na.rm = TRUE)
looksRaw <- max(combinedCheck, na.rm = TRUE) > 50

if (hasNegative) {
  cat("Data already normalized/log-scale (negative values present). No transform applied.\n")
} else if (looksRaw) {
  expr_A_hub[] <- log2(expr_A_hub + 1)
  expr_B_hub[] <- log2(expr_B_hub + 1)
  cat("Applied log2(x+1) transform.\n")
} else {
  cat("Data already log-scale — no transform applied.\n")
}

############################################################
## Long format — Status = target (0 = Normal, 1 = Affected)
############################################################

longData <- do.call(rbind, lapply(present_hub, function(g){
  data.frame(
    Gene = g,
    Sample = c(rownames(expr_A_hub), rownames(expr_B_hub)),
    Dataset = c(rep("GSE33630", nrow(expr_A_hub)), rep("GSE76894", nrow(expr_B_hub))),
    Target = c(dataA$target, dataB$target),
    Expression = c(expr_A_hub[[g]], expr_B_hub[[g]])
  )
}))

longData$Status <- factor(longData$Target, levels = c(0, 1), labels = c("Normal", "Affected"))

nBefore <- nrow(longData)
longData <- longData[is.finite(longData$Expression) & !is.na(longData$Status), ]
nAfter <- nrow(longData)
if (nBefore != nAfter) cat("\nRemoved", nBefore - nAfter, "rows with non-finite expression or unmapped target.\n")
if (nAfter == 0) stop("No valid rows remain after cleaning — check target coding (expects 0/1).")

############################################################
## Bootstrap function — robust
############################################################

bootstrap_mean_ci <- function(x, nboot = 2000, seed = 1){
  x <- x[is.finite(x)]
  n <- length(x)
  if (n < 2) return(data.frame(Mean = ifelse(n==1, x, NA), CI_lower = NA, CI_upper = NA, N = n))
  set.seed(seed)
  boot_means <- replicate(nboot, mean(sample(x, n, replace = TRUE)))
  boot_means <- boot_means[is.finite(boot_means)]
  if (length(boot_means) == 0) return(data.frame(Mean = mean(x), CI_lower = NA, CI_upper = NA, N = n))
  data.frame(Mean = mean(x),
             CI_lower = quantile(boot_means, 0.025, na.rm = TRUE),
             CI_upper = quantile(boot_means, 0.975, na.rm = TRUE),
             N = n)
}

############################################################
## Colour scheme (Normal vs Affected, journal-friendly)
############################################################

fillCols <- c("Normal" = "#4C8FC9", "Affected" = "#E4655C")

############################################################
## Loop: build TWO SEPARATE figures per gene (A and B)
## Panel A/B both faceted by Dataset (GSE33630 vs GSE76894)
############################################################

summaryList <- list()
genePlots_A <- list()
genePlots_B <- list()

for (g in present_hub) {

  geneData <- subset(longData, Gene == g)
  if (nrow(geneData) == 0) { cat("Skipping", g, "- no data\n"); next }

  ##########################################################
  ## Figure A — distribution (boxplot + points), Affected vs Normal
  ##########################################################

  pA <- ggplot(geneData, aes(x = Status, y = Expression, fill = Status)) +
    geom_boxplot(alpha = 0.75, outlier.shape = NA, width = 0.55, linewidth = 0.5) +
    geom_jitter(width = 0.10, size = 2.2, alpha = 0.9, shape = 21, colour = "black", stroke = 0.4) +
    facet_wrap(~ Dataset) +
    scale_fill_manual(values = fillCols) +
    theme_pub(12) +
    labs(title = paste0(g, " — Expression Distribution"), x = "", y = "Expression")

  genePlots_A[[g]] <- pA

  outPathA_png <- file.path(OUTDIR, paste0(g, "_A_Distribution.png"))
  outPathA_pdf <- file.path(OUTDIR, paste0(g, "_A_Distribution.pdf"))
  ggsave(outPathA_png, plot = pA, width = 5.5, height = 4.6, dpi = 320, bg = "white")
  ggsave(outPathA_pdf, plot = pA, width = 5.5, height = 4.6, bg = "white")
  cat("Saved distribution figure for", g, "->", outPathA_png, "\n")

  ##########################################################
  ## Figure B — bootstrap CI, Affected vs Normal per Dataset
  ##########################################################

  ciData <- do.call(rbind, lapply(unique(geneData$Dataset), function(ds){
    do.call(rbind, lapply(c("Normal", "Affected"), function(st){
      x <- geneData$Expression[geneData$Dataset == ds & geneData$Status == st]
      seed_val <- which(unique(geneData$Dataset) == ds) * 10 + ifelse(st == "Normal", 1, 2)
      ci <- bootstrap_mean_ci(x, seed = seed_val)
      data.frame(Dataset = ds, Status = st, ci)
    }))
  }))

  pB <- ggplot(ciData, aes(x = Status, y = Mean, fill = Status)) +
    geom_col(width = 0.55, colour = "black", alpha = 0.85, linewidth = 0.5) +
    geom_errorbar(aes(ymin = CI_lower, ymax = CI_upper), width = 0.15, linewidth = 0.7, na.rm = TRUE) +
    geom_text(aes(label = paste0("n=", N), y = pmax(CI_upper, Mean, na.rm = TRUE)),
              vjust = -1.2, size = 3.4, fontface = "italic") +
    facet_wrap(~ Dataset) +
    scale_fill_manual(values = fillCols) +
    theme_pub(12) +
    labs(title = paste0(g, " — Bootstrap 95% CI"), x = "", y = "Mean Expression") +
    expand_limits(y = max(ciData$CI_upper, na.rm = TRUE) * 1.15)

  genePlots_B[[g]] <- pB

  outPathB_png <- file.path(OUTDIR, paste0(g, "_B_BootstrapCI.png"))
  outPathB_pdf <- file.path(OUTDIR, paste0(g, "_B_BootstrapCI.pdf"))
  ggsave(outPathB_png, plot = pB, width = 5.5, height = 4.6, dpi = 320, bg = "white")
  ggsave(outPathB_pdf, plot = pB, width = 5.5, height = 4.6, bg = "white")
  cat("Saved bootstrap CI figure for", g, "->", outPathB_png, "\n")

  summaryList[[g]] <- data.frame(
    Gene = g,
    Dataset = ciData$Dataset,
    Status = ciData$Status,
    N = ciData$N,
    Mean_Expression = round(ciData$Mean, 4),
    Bootstrap_CI_Lower = round(ciData$CI_lower, 4),
    Bootstrap_CI_Upper = round(ciData$CI_upper, 4)
  )
}

############################################################
## Master overview figure — all hub genes, A and B panels
## interleaved (12 panels for 6 genes), stacked N rows x 1
############################################################

if (length(genePlots_A) > 0) {

  interleaved <- list()
  for (g in present_hub) {
    if (!is.null(genePlots_A[[g]])) interleaved[[paste0(g, "_A")]] <- genePlots_A[[g]]
    if (!is.null(genePlots_B[[g]])) interleaved[[paste0(g, "_B")]] <- genePlots_B[[g]]
  }

  masterFig <- wrap_plots(interleaved, ncol = 2) +
    plot_annotation(
      title = "Hub Gene Expression \u2014 Bootstrap Analysis (GSE33630 vs GSE76894, Affected vs Normal)",
      theme = theme(plot.title = element_text(face = "bold", size = 18, hjust = 0.5))
    )

  nRows <- length(present_hub)

  ggsave(file.path(OUTDIR, "AllHubGenes_Overview_6genes.png"),
         plot = masterFig, width = 11, height = 4.6 * nRows, dpi = 300, bg = "white", limitsize = FALSE)

  ggsave(file.path(OUTDIR, "AllHubGenes_Overview_6genes.pdf"),
         plot = masterFig, width = 11, height = 4.6 * nRows, bg = "white", limitsize = FALSE)

  cat("\nMaster overview figure saved: AllHubGenes_Overview_6genes.png / .pdf\n")
}

############################################################
## Master summary table
############################################################

masterSummary <- do.call(rbind, summaryList)
rownames(masterSummary) <- NULL

write.csv(masterSummary, file.path(TABLEDIR, "Table_HubGenes_Bootstrap_Summary_6genes.csv"), row.names = FALSE)

cat("\n============================================================\n")
cat("Master summary table:\n")
print(masterSummary)
cat("============================================================\n")

cat("\nALL OUTPUTS SAVED TO:", OUTDIR, "\n")
cat("Per-gene SEPARATED figures (12 total for 6 genes):\n")
for (g in present_hub) {
  cat("-", paste0(g, "_A_Distribution.png / .pdf"), "\n")
  cat("-", paste0(g, "_B_BootstrapCI.png / .pdf"), "\n")
}
cat("Master overview: AllHubGenes_Overview_6genes.png / .pdf\n")
cat("Table: Tables/Table_HubGenes_Bootstrap_Summary_6genes.csv\n")

GSE33630: 105 samples, 24503 genes (target column: target )
GSE76894: 103 samples, 8637 genes (target column: target )

GSE33630 target distribution:

  0   1 
  1 104 
GSE76894 target distribution:

  0   1 
  1 102 

Hub genes available in both datasets: 6 of 6 — MAP2K7, PPARA, RBCK1, NFATC2, PLXNA4, STAMBP 

Value range — min: 3.671  max: 11.266 
Data already log-scale — no transform applied.
Saved distribution figure for MAP2K7 -> /content/drive/MyDrive/Bootstrap anlysis/MAP2K7_A_Distribution.png 
Saved bootstrap CI figure for MAP2K7 -> /content/drive/MyDrive/Bootstrap anlysis/MAP2K7_B_BootstrapCI.png 
Saved distribution figure for PPARA -> /content/drive/MyDrive/Bootstrap anlysis/PPARA_A_Distribution.png 
Saved bootstrap CI figure for PPARA -> /content/drive/MyDrive/Bootstrap anlysis/PPARA_B_BootstrapCI.png 
Saved distribution figure for RBCK1 -> /content/drive/MyDrive/Bootstrap anlysis/RBCK1_A_Distribution.png 
Saved bootstrap CI figure for RBCK1 -> /content/drive/MyDrive/Bootstr

In addition: There were 25 warnings (use warnings() to see them)


#Final Code

In [16]:
%%R
############################################################
## Individual Hub Gene Bootstrap Analysis — PUBLICATION READY
## CORRECTED FOR WIDE-FORMAT INPUT:
##   rows = samples, columns = genes, LAST column = target (0/1)
## Each gene: ONE combined 2-panel figure (A = distribution,
## B = bootstrap CI), grouped by Affected vs Normal, faceted by dataset.
## Plus one master overview figure with all hub genes.
## Output: /content/drive/MyDrive/Bootstrap anlysis
############################################################

if (!requireNamespace("patchwork", quietly = TRUE)) install.packages("patchwork", repos = "https://cloud.r-project.org")

suppressPackageStartupMessages({
  library(data.table); library(ggplot2); library(patchwork)
})
options(stringsAsFactors = FALSE)

OUTDIR <- "/content/drive/MyDrive/Bootstrap anlysis"
if (!dir.exists(OUTDIR)) dir.create(OUTDIR, recursive = TRUE)

TABLEDIR <- file.path(OUTDIR, "Tables")
if (!dir.exists(TABLEDIR)) dir.create(TABLEDIR, recursive = TRUE)

TRAIN_EXPR <- "/content/drive/MyDrive/Bootstrap anlysis/datasets/GSE33630_matrix_Transpose_labeled.csv"
TEST_EXPR  <- "/content/drive/MyDrive/Bootstrap anlysis/datasets/GSE76894_matrix_Transpose_labeled.csv"

## ---- 6 hub genes ----
hub_genes_top5 <- c("MAP2K7","PPARA","RBCK1","NFATC2","PLXNA4","STAMBP")

############################################################
## Publication theme — clean, journal-style
############################################################

theme_pub <- function(base_size = 13){
  theme_bw(base_size = base_size) +
    theme(
      panel.grid.minor = element_blank(),
      panel.grid.major.x = element_blank(),
      plot.title = element_text(face = "bold", size = base_size + 1, hjust = 0.5),
      axis.title = element_text(face = "bold", size = base_size),
      axis.text = element_text(size = base_size - 1, colour = "black"),
      legend.position = "none",
      plot.margin = margin(10, 14, 10, 10)
    )
}

############################################################
## Load WIDE-format expression data
## rows = samples, columns = genes, LAST column = target (0/1)
## optionally the FIRST column may be a non-numeric sample ID
############################################################

read_expr_wide <- function(path){
  dt <- fread(path)

  target_col_idx <- ncol(dt)
  target_col_name <- names(dt)[target_col_idx]
  target_raw <- dt[[target_col_idx]]
  dt[[target_col_idx]] <- NULL

  df <- as.data.frame(dt)

  first_col <- df[[1]]
  first_is_id <- is.character(first_col) ||
    all(is.na(suppressWarnings(as.numeric(as.character(first_col)))))

  if (first_is_id) {
    samp_ids <- make.unique(as.character(first_col))
    df[[1]] <- NULL
  } else {
    samp_ids <- paste0("Sample_", seq_len(nrow(df)))
  }

  df[] <- lapply(df, function(col) suppressWarnings(as.numeric(as.character(col))))
  rownames(df) <- samp_ids

  target <- suppressWarnings(as.numeric(as.character(target_raw)))

  list(expr = df, target = target, target_col_name = target_col_name, sample_ids = samp_ids)
}

dataA <- read_expr_wide(TRAIN_EXPR)   # GSE33630
dataB <- read_expr_wide(TEST_EXPR)    # GSE76894

expr_A <- dataA$expr
expr_B <- dataB$expr

cat("GSE33630:", nrow(expr_A), "samples,", ncol(expr_A), "genes",
    "(target column:", dataA$target_col_name, ")\n")
cat("GSE76894:", nrow(expr_B), "samples,", ncol(expr_B), "genes",
    "(target column:", dataB$target_col_name, ")\n")

cat("\nGSE33630 target distribution:\n"); print(table(dataA$target, useNA = "ifany"))
cat("GSE76894 target distribution:\n"); print(table(dataB$target, useNA = "ifany"))

present_hub <- intersect(hub_genes_top5, intersect(colnames(expr_A), colnames(expr_B)))
cat("\nHub genes available in both datasets:", length(present_hub), "of", length(hub_genes_top5), "—",
    paste(present_hub, collapse=", "), "\n")
missing_hub <- setdiff(hub_genes_top5, present_hub)
if (length(missing_hub) > 0) cat("Missing:", paste(missing_hub, collapse=", "), "\n")

if (length(present_hub) == 0) {
  stop("No hub genes matched any column names. Check for symbol-case mismatches, ",
       "prefixes (e.g. 'X' prepended by R to numeric-looking names), or whitespace ",
       "in the CSV header — print colnames(expr_A)[1:20] to inspect.")
}

expr_A_hub <- expr_A[, present_hub, drop = FALSE]
expr_B_hub <- expr_B[, present_hub, drop = FALSE]
combinedCheck <- as.matrix(rbind(expr_A_hub, expr_B_hub))

cat("\nValue range — min:", round(min(combinedCheck, na.rm = TRUE), 3),
    " max:", round(max(combinedCheck, na.rm = TRUE), 3), "\n")

hasNegative <- any(combinedCheck < 0, na.rm = TRUE)
looksRaw <- max(combinedCheck, na.rm = TRUE) > 50

if (hasNegative) {
  cat("Data already normalized/log-scale (negative values present). No transform applied.\n")
} else if (looksRaw) {
  expr_A_hub[] <- log2(expr_A_hub + 1)
  expr_B_hub[] <- log2(expr_B_hub + 1)
  cat("Applied log2(x+1) transform.\n")
} else {
  cat("Data already log-scale — no transform applied.\n")
}

############################################################
## Long format — Status = target (0 = Normal, 1 = Affected)
############################################################

longData <- do.call(rbind, lapply(present_hub, function(g){
  data.frame(
    Gene = g,
    Sample = c(rownames(expr_A_hub), rownames(expr_B_hub)),
    Dataset = c(rep("GSE33630", nrow(expr_A_hub)), rep("GSE76894", nrow(expr_B_hub))),
    Target = c(dataA$target, dataB$target),
    Expression = c(expr_A_hub[[g]], expr_B_hub[[g]])
  )
}))

longData$Status <- factor(longData$Target, levels = c(0, 1), labels = c("Normal", "Affected"))

nBefore <- nrow(longData)
longData <- longData[is.finite(longData$Expression) & !is.na(longData$Status), ]
nAfter <- nrow(longData)
if (nBefore != nAfter) cat("\nRemoved", nBefore - nAfter, "rows with non-finite expression or unmapped target.\n")
if (nAfter == 0) stop("No valid rows remain after cleaning — check target coding (expects 0/1).")

############################################################
## Bootstrap function — robust
############################################################

bootstrap_mean_ci <- function(x, nboot = 2000, seed = 1){
  x <- x[is.finite(x)]
  n <- length(x)
  if (n < 2) return(data.frame(Mean = ifelse(n==1, x, NA), CI_lower = NA, CI_upper = NA, N = n))
  set.seed(seed)
  boot_means <- replicate(nboot, mean(sample(x, n, replace = TRUE)))
  boot_means <- boot_means[is.finite(boot_means)]
  if (length(boot_means) == 0) return(data.frame(Mean = mean(x), CI_lower = NA, CI_upper = NA, N = n))
  data.frame(Mean = mean(x),
             CI_lower = quantile(boot_means, 0.025, na.rm = TRUE),
             CI_upper = quantile(boot_means, 0.975, na.rm = TRUE),
             N = n)
}

############################################################
## Colour scheme (Normal vs Affected, journal-friendly)
############################################################

fillCols <- c("Normal" = "#4C8FC9", "Affected" = "#E4655C")

############################################################
## Loop: build combined 2-panel figure per gene
## Panel A/B both faceted by Dataset (GSE33630 vs GSE76894)
############################################################

summaryList <- list()
genePlots <- list()

for (g in present_hub) {

  geneData <- subset(longData, Gene == g)
  if (nrow(geneData) == 0) { cat("Skipping", g, "- no data\n"); next }

  ##########################################################
  ## Panel A — distribution (boxplot + points), Affected vs Normal
  ##########################################################

  pA <- ggplot(geneData, aes(x = Status, y = Expression, fill = Status)) +
    geom_boxplot(alpha = 0.75, outlier.shape = NA, width = 0.55, linewidth = 0.5) +
    geom_jitter(width = 0.10, size = 2.2, alpha = 0.9, shape = 21, colour = "black", stroke = 0.4) +
    facet_wrap(~ Dataset) +
    scale_fill_manual(values = fillCols) +
    theme_pub(12) +
    labs(title = "A. Expression Distribution", x = "", y = "Expression")

  ##########################################################
  ## Panel B — bootstrap CI, Affected vs Normal per Dataset
  ##########################################################

  ciData <- do.call(rbind, lapply(unique(geneData$Dataset), function(ds){
    do.call(rbind, lapply(c("Normal", "Affected"), function(st){
      x <- geneData$Expression[geneData$Dataset == ds & geneData$Status == st]
      seed_val <- which(unique(geneData$Dataset) == ds) * 10 + ifelse(st == "Normal", 1, 2)
      ci <- bootstrap_mean_ci(x, seed = seed_val)
      data.frame(Dataset = ds, Status = st, ci)
    }))
  }))

  pB <- ggplot(ciData, aes(x = Status, y = Mean, fill = Status)) +
    geom_col(width = 0.55, colour = "black", alpha = 0.85, linewidth = 0.5) +
    geom_errorbar(aes(ymin = CI_lower, ymax = CI_upper), width = 0.15, linewidth = 0.7, na.rm = TRUE) +
    geom_text(aes(label = paste0("n=", N), y = pmax(CI_upper, Mean, na.rm = TRUE)),
              vjust = -1.2, size = 3.4, fontface = "italic") +
    facet_wrap(~ Dataset) +
    scale_fill_manual(values = fillCols) +
    theme_pub(12) +
    labs(title = "B. Bootstrap 95% CI", x = "", y = "Mean Expression") +
    expand_limits(y = max(ciData$CI_upper, na.rm = TRUE) * 1.15)

  ##########################################################
  ## Combine A + B side by side, with a gene-level title
  ##########################################################

  combinedPlot <- (pA | pB) +
    plot_annotation(
      title = g,
      theme = theme(plot.title = element_text(face = "bold.italic", size = 16, hjust = 0.5))
    )

  genePlots[[g]] <- combinedPlot

  outPath <- file.path(OUTDIR, paste0(g, "_Combined_Figure.png"))
  ggsave(outPath, plot = combinedPlot, width = 9.5, height = 4.6, dpi = 320, bg = "white")

  outPathPDF <- file.path(OUTDIR, paste0(g, "_Combined_Figure.pdf"))
  ggsave(outPathPDF, plot = combinedPlot, width = 9.5, height = 4.6, bg = "white")

  cat("Saved combined figure for", g, "->", outPath, "\n")

  summaryList[[g]] <- data.frame(
    Gene = g,
    Dataset = ciData$Dataset,
    Status = ciData$Status,
    N = ciData$N,
    Mean_Expression = round(ciData$Mean, 4),
    Bootstrap_CI_Lower = round(ciData$CI_lower, 4),
    Bootstrap_CI_Upper = round(ciData$CI_upper, 4)
  )
}

############################################################
## Master overview figure — all hub genes stacked (N rows x 1)
############################################################

if (length(genePlots) > 0) {
  masterFig <- wrap_plots(genePlots, ncol = 1) +
    plot_annotation(
      title = "Hub Gene Expression \u2014 Bootstrap Analysis (GSE33630 vs GSE76894, Affected vs Normal)",
      theme = theme(plot.title = element_text(face = "bold", size = 18, hjust = 0.5))
    )

  ggsave(file.path(OUTDIR, "AllHubGenes_Combined_Overview_6genes.png"),
         plot = masterFig, width = 10, height = 4.6 * length(genePlots), dpi = 300, bg = "white", limitsize = FALSE)

  ggsave(file.path(OUTDIR, "AllHubGenes_Combined_Overview_6genes.pdf"),
         plot = masterFig, width = 10, height = 4.6 * length(genePlots), bg = "white", limitsize = FALSE)

  cat("\nMaster overview figure saved: AllHubGenes_Combined_Overview_6genes.png / .pdf\n")
}

############################################################
## Master summary table
############################################################

masterSummary <- do.call(rbind, summaryList)
rownames(masterSummary) <- NULL

write.csv(masterSummary, file.path(TABLEDIR, "Table_HubGenes_Bootstrap_Summary_6genes.csv"), row.names = FALSE)

cat("\n============================================================\n")
cat("Master summary table:\n")
print(masterSummary)
cat("============================================================\n")

cat("\nALL OUTPUTS SAVED TO:", OUTDIR, "\n")
cat("Per-gene combined figures (A+B in one image):\n")
for (g in present_hub) cat("-", paste0(g, "_Combined_Figure.png / .pdf"), "\n")
cat("Master overview: AllHubGenes_Combined_Overview_6genes.png / .pdf\n")
cat("Table: Tables/Table_HubGenes_Bootstrap_Summary_6genes.csv\n")

GSE33630: 105 samples, 24503 genes (target column: target )
GSE76894: 103 samples, 8637 genes (target column: target )

GSE33630 target distribution:

  0   1 
  1 104 
GSE76894 target distribution:

  0   1 
  1 102 

Hub genes available in both datasets: 6 of 6 — MAP2K7, PPARA, RBCK1, NFATC2, PLXNA4, STAMBP 

Value range — min: 3.671  max: 11.266 
Data already log-scale — no transform applied.
Saved combined figure for MAP2K7 -> /content/drive/MyDrive/Bootstrap anlysis/MAP2K7_Combined_Figure.png 
Saved combined figure for PPARA -> /content/drive/MyDrive/Bootstrap anlysis/PPARA_Combined_Figure.png 
Saved combined figure for RBCK1 -> /content/drive/MyDrive/Bootstrap anlysis/RBCK1_Combined_Figure.png 
Saved combined figure for NFATC2 -> /content/drive/MyDrive/Bootstrap anlysis/NFATC2_Combined_Figure.png 
Saved combined figure for PLXNA4 -> /content/drive/MyDrive/Bootstrap anlysis/PLXNA4_Combined_Figure.png 
Saved combined figure for STAMBP -> /content/drive/MyDrive/Bootstrap anlysis/STA

In addition: Warning message:
In grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y,  :
  for 'Hub Gene Expression — Bootstrap Analysis (GSE33630 vs GSE76894, Affected vs Normal)' in 'mbcsToSbcs': - substituted for — (U+2014)


In [17]:
%%R
############################################################
## Individual Hub Gene Bootstrap Analysis — PUBLICATION READY
## CORRECTED FOR WIDE-FORMAT INPUT:
##   rows = samples, columns = genes, LAST column = target (0/1)
## Each gene: ONE combined 2-panel figure (A = distribution,
## B = bootstrap CI), grouped/colored BY DATASET
## (GSE33630 vs GSE76894) — NOT by Normal/Affected.
## Plus one master overview figure with all hub genes.
## Output: /content/drive/MyDrive/Bootstrap anlysis
############################################################

if (!requireNamespace("patchwork", quietly = TRUE)) install.packages("patchwork", repos = "https://cloud.r-project.org")

suppressPackageStartupMessages({
  library(data.table); library(ggplot2); library(patchwork)
})
options(stringsAsFactors = FALSE)

OUTDIR <- "/content/drive/MyDrive/Bootstrap anlysis/Final_Figure_Table"
if (!dir.exists(OUTDIR)) dir.create(OUTDIR, recursive = TRUE)

TABLEDIR <- file.path(OUTDIR, "Tables")
if (!dir.exists(TABLEDIR)) dir.create(TABLEDIR, recursive = TRUE)

TRAIN_EXPR <- "/content/drive/MyDrive/Bootstrap anlysis/datasets/GSE33630_matrix_Transpose_labeled.csv"
TEST_EXPR  <- "/content/drive/MyDrive/Bootstrap anlysis/datasets/GSE76894_matrix_Transpose_labeled.csv"

## ---- 6 hub genes ----
hub_genes_top5 <- c("MAP2K7","PPARA","RBCK1","NFATC2","PLXNA4","STAMBP")

############################################################
## Publication theme — clean, journal-style
############################################################

theme_pub <- function(base_size = 13){
  theme_bw(base_size = base_size) +
    theme(
      panel.grid.minor = element_blank(),
      panel.grid.major.x = element_blank(),
      plot.title = element_text(face = "bold", size = base_size + 1, hjust = 0.5),
      axis.title = element_text(face = "bold", size = base_size),
      axis.text = element_text(size = base_size - 1, colour = "black"),
      legend.position = "none",
      plot.margin = margin(10, 14, 10, 10)
    )
}

############################################################
## Load WIDE-format expression data
## rows = samples, columns = genes, LAST column = target (0/1)
## optionally the FIRST column may be a non-numeric sample ID
############################################################

read_expr_wide <- function(path){
  dt <- fread(path)

  target_col_idx <- ncol(dt)
  target_col_name <- names(dt)[target_col_idx]
  target_raw <- dt[[target_col_idx]]
  dt[[target_col_idx]] <- NULL

  df <- as.data.frame(dt)

  first_col <- df[[1]]
  first_is_id <- is.character(first_col) ||
    all(is.na(suppressWarnings(as.numeric(as.character(first_col)))))

  if (first_is_id) {
    samp_ids <- make.unique(as.character(first_col))
    df[[1]] <- NULL
  } else {
    samp_ids <- paste0("Sample_", seq_len(nrow(df)))
  }

  df[] <- lapply(df, function(col) suppressWarnings(as.numeric(as.character(col))))
  rownames(df) <- samp_ids

  target <- suppressWarnings(as.numeric(as.character(target_raw)))

  list(expr = df, target = target, target_col_name = target_col_name, sample_ids = samp_ids)
}

dataA <- read_expr_wide(TRAIN_EXPR)   # GSE33630
dataB <- read_expr_wide(TEST_EXPR)    # GSE76894

expr_A <- dataA$expr
expr_B <- dataB$expr

cat("GSE33630:", nrow(expr_A), "samples,", ncol(expr_A), "genes",
    "(target column:", dataA$target_col_name, ")\n")
cat("GSE76894:", nrow(expr_B), "samples,", ncol(expr_B), "genes",
    "(target column:", dataB$target_col_name, ")\n")

present_hub <- intersect(hub_genes_top5, intersect(colnames(expr_A), colnames(expr_B)))
cat("\nHub genes available in both datasets:", length(present_hub), "of", length(hub_genes_top5), "—",
    paste(present_hub, collapse=", "), "\n")
missing_hub <- setdiff(hub_genes_top5, present_hub)
if (length(missing_hub) > 0) cat("Missing:", paste(missing_hub, collapse=", "), "\n")

if (length(present_hub) == 0) {
  stop("No hub genes matched any column names. Check for symbol-case mismatches, ",
       "prefixes (e.g. 'X' prepended by R to numeric-looking names), or whitespace ",
       "in the CSV header — print colnames(expr_A)[1:20] to inspect.")
}

expr_A_hub <- expr_A[, present_hub, drop = FALSE]
expr_B_hub <- expr_B[, present_hub, drop = FALSE]
combinedCheck <- as.matrix(rbind(expr_A_hub, expr_B_hub))

cat("\nValue range — min:", round(min(combinedCheck, na.rm = TRUE), 3),
    " max:", round(max(combinedCheck, na.rm = TRUE), 3), "\n")

hasNegative <- any(combinedCheck < 0, na.rm = TRUE)
looksRaw <- max(combinedCheck, na.rm = TRUE) > 50

if (hasNegative) {
  cat("Data already normalized/log-scale (negative values present). No transform applied.\n")
} else if (looksRaw) {
  expr_A_hub[] <- log2(expr_A_hub + 1)
  expr_B_hub[] <- log2(expr_B_hub + 1)
  cat("Applied log2(x+1) transform.\n")
} else {
  cat("Data already log-scale — no transform applied.\n")
}

############################################################
## Long format — grouped by Dataset (Normal/Affected DROPPED)
############################################################

longData <- do.call(rbind, lapply(present_hub, function(g){
  data.frame(
    Gene = g,
    Sample = c(rownames(expr_A_hub), rownames(expr_B_hub)),
    Dataset = c(rep("GSE33630", nrow(expr_A_hub)), rep("GSE76894", nrow(expr_B_hub))),
    Expression = c(expr_A_hub[[g]], expr_B_hub[[g]])
  )
}))

longData$Dataset <- factor(longData$Dataset, levels = c("GSE33630", "GSE76894"))

nBefore <- nrow(longData)
longData <- longData[is.finite(longData$Expression), ]
nAfter <- nrow(longData)
if (nBefore != nAfter) cat("\nRemoved", nBefore - nAfter, "rows with non-finite expression.\n")
if (nAfter == 0) stop("No valid rows remain after cleaning.")

############################################################
## Bootstrap function — robust
############################################################

bootstrap_mean_ci <- function(x, nboot = 2000, seed = 1){
  x <- x[is.finite(x)]
  n <- length(x)
  if (n < 2) return(data.frame(Mean = ifelse(n==1, x, NA), CI_lower = NA, CI_upper = NA, N = n))
  set.seed(seed)
  boot_means <- replicate(nboot, mean(sample(x, n, replace = TRUE)))
  boot_means <- boot_means[is.finite(boot_means)]
  if (length(boot_means) == 0) return(data.frame(Mean = mean(x), CI_lower = NA, CI_upper = NA, N = n))
  data.frame(Mean = mean(x),
             CI_lower = quantile(boot_means, 0.025, na.rm = TRUE),
             CI_upper = quantile(boot_means, 0.975, na.rm = TRUE),
             N = n)
}

############################################################
## Colour scheme (by Dataset — same red/blue as sample picture)
############################################################

fillCols <- c("GSE33630" = "#E4655C", "GSE76894" = "#4C8FC9")

############################################################
## Loop: build combined 2-panel figure per gene, grouped by Dataset
############################################################

summaryList <- list()
genePlots <- list()

for (g in present_hub) {

  geneData <- subset(longData, Gene == g)
  if (nrow(geneData) == 0) { cat("Skipping", g, "- no data\n"); next }

  ##########################################################
  ## Panel A — distribution (boxplot + points), by Dataset
  ##########################################################

  pA <- ggplot(geneData, aes(x = Dataset, y = Expression, fill = Dataset)) +
    geom_boxplot(alpha = 0.75, outlier.shape = NA, width = 0.55, linewidth = 0.5) +
    geom_jitter(width = 0.10, size = 2.2, alpha = 0.9, shape = 21, colour = "black", stroke = 0.4) +
    scale_fill_manual(values = fillCols) +
    theme_pub(12) +
    labs(title = "A. Expression Distribution", x = "", y = "Expression")

  ##########################################################
  ## Panel B — bootstrap CI, by Dataset
  ##########################################################

  ciData <- do.call(rbind, lapply(levels(geneData$Dataset), function(ds){
    x <- geneData$Expression[geneData$Dataset == ds]
    seed_val <- which(levels(geneData$Dataset) == ds)
    ci <- bootstrap_mean_ci(x, seed = seed_val)
    data.frame(Dataset = ds, ci)
  }))
  ciData$Dataset <- factor(ciData$Dataset, levels = levels(geneData$Dataset))

  pB <- ggplot(ciData, aes(x = Dataset, y = Mean, fill = Dataset)) +
    geom_col(width = 0.55, colour = "black", alpha = 0.85, linewidth = 0.5) +
    geom_errorbar(aes(ymin = CI_lower, ymax = CI_upper), width = 0.15, linewidth = 0.7, na.rm = TRUE) +
    geom_text(aes(label = paste0("n=", N), y = pmax(CI_upper, Mean, na.rm = TRUE)),
              vjust = -1.2, size = 3.4, fontface = "italic") +
    scale_fill_manual(values = fillCols) +
    theme_pub(12) +
    labs(title = "B. Bootstrap 95% CI", x = "", y = "Mean Expression") +
    expand_limits(y = max(ciData$CI_upper, na.rm = TRUE) * 1.15)

  ##########################################################
  ## Combine A + B side by side, with a gene-level title
  ##########################################################

  combinedPlot <- (pA | pB) +
    plot_annotation(
      title = g,
      theme = theme(plot.title = element_text(face = "bold.italic", size = 16, hjust = 0.5))
    )

  genePlots[[g]] <- combinedPlot

  outPath <- file.path(OUTDIR, paste0(g, "_Combined_Figure.png"))
  ggsave(outPath, plot = combinedPlot, width = 9.5, height = 4.6, dpi = 320, bg = "white")

  outPathPDF <- file.path(OUTDIR, paste0(g, "_Combined_Figure.pdf"))
  ggsave(outPathPDF, plot = combinedPlot, width = 9.5, height = 4.6, bg = "white")

  cat("Saved combined figure for", g, "->", outPath, "\n")

  summaryList[[g]] <- data.frame(
    Gene = g,
    Dataset = ciData$Dataset,
    N = ciData$N,
    Mean_Expression = round(ciData$Mean, 4),
    Bootstrap_CI_Lower = round(ciData$CI_lower, 4),
    Bootstrap_CI_Upper = round(ciData$CI_upper, 4)
  )
}

############################################################
## Master overview figure — all hub genes stacked (N rows x 1)
############################################################

if (length(genePlots) > 0) {
  masterFig <- wrap_plots(genePlots, ncol = 1) +
    plot_annotation(
      title = "Hub Gene Expression \u2014 Bootstrap Analysis by Dataset (GSE33630 vs GSE76894)",
      theme = theme(plot.title = element_text(face = "bold", size = 18, hjust = 0.5))
    )

  ggsave(file.path(OUTDIR, "AllHubGenes_Combined_Overview_byDataset.png"),
         plot = masterFig, width = 10, height = 4.6 * length(genePlots), dpi = 300, bg = "white", limitsize = FALSE)

  ggsave(file.path(OUTDIR, "AllHubGenes_Combined_Overview_byDataset.pdf"),
         plot = masterFig, width = 10, height = 4.6 * length(genePlots), bg = "white", limitsize = FALSE)

  cat("\nMaster overview figure saved: AllHubGenes_Combined_Overview_byDataset.png / .pdf\n")
}

############################################################
## Master summary table
############################################################

masterSummary <- do.call(rbind, summaryList)
rownames(masterSummary) <- NULL

write.csv(masterSummary, file.path(TABLEDIR, "Table_HubGenes_Bootstrap_Summary_byDataset.csv"), row.names = FALSE)

cat("\n============================================================\n")
cat("Master summary table:\n")
print(masterSummary)
cat("============================================================\n")

cat("\nALL OUTPUTS SAVED TO:", OUTDIR, "\n")
cat("Per-gene combined figures (A+B in one image, grouped by Dataset):\n")
for (g in present_hub) cat("-", paste0(g, "_Combined_Figure.png / .pdf"), "\n")
cat("Master overview: AllHubGenes_Combined_Overview_byDataset.png / .pdf\n")
cat("Table: Tables/Table_HubGenes_Bootstrap_Summary_byDataset.csv\n")

GSE33630: 105 samples, 24503 genes (target column: target )
GSE76894: 103 samples, 8637 genes (target column: target )

Hub genes available in both datasets: 6 of 6 — MAP2K7, PPARA, RBCK1, NFATC2, PLXNA4, STAMBP 

Value range — min: 3.671  max: 11.266 
Data already log-scale — no transform applied.
Saved combined figure for MAP2K7 -> /content/drive/MyDrive/Bootstrap anlysis/Final_Figure_Table/MAP2K7_Combined_Figure.png 
Saved combined figure for PPARA -> /content/drive/MyDrive/Bootstrap anlysis/Final_Figure_Table/PPARA_Combined_Figure.png 
Saved combined figure for RBCK1 -> /content/drive/MyDrive/Bootstrap anlysis/Final_Figure_Table/RBCK1_Combined_Figure.png 
Saved combined figure for NFATC2 -> /content/drive/MyDrive/Bootstrap anlysis/Final_Figure_Table/NFATC2_Combined_Figure.png 
Saved combined figure for PLXNA4 -> /content/drive/MyDrive/Bootstrap anlysis/Final_Figure_Table/PLXNA4_Combined_Figure.png 
Saved combined figure for STAMBP -> /content/drive/MyDrive/Bootstrap anlysis/Final_F

In addition: Warning message:
In grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y,  :
  for 'Hub Gene Expression — Bootstrap Analysis by Dataset (GSE33630 vs GSE76894)' in 'mbcsToSbcs': - substituted for — (U+2014)
